# The Hugging Face Ecosystem for Genomics

**Day 1 Afternoon - Session 2**

**Author:** Ikram Ullah, KAUST Bioinformatics Platform

---

## Overview

The **Hugging Face ecosystem** provides tools for accessing, using, and sharing machine learning models. In this notebook, we'll explore:

1. **Model Hub** - Browse and discover genomic models
2. **Inference API** - Run models in the cloud without local GPU
3. **Local vs API inference** - Understand the trade-offs
4. **Datasets Hub** - Access genomic benchmark datasets

---

## Learning Objectives

1. Navigate the Hugging Face Model Hub to find genomic models
2. Read and interpret Model Cards
3. Use the Inference API for quick prototyping
4. Compare local GPU inference vs cloud API
5. Load genomic datasets from the Datasets Hub

---

## Prerequisites

- Hugging Face account (free): https://huggingface.co/join
- HF API token set as `HF_TOKEN` environment variable

---

## Setup

In [ ]:
import torch
import numpy as np
import os
import requests
import time
import pandas as pd
from transformers import AutoTokenizer, AutoModel, AutoModelForMaskedLM, AutoConfig
from IPython.display import display, HTML, Markdown
import torch.nn.functional as F

# Device setup
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"PyTorch version: {torch.__version__}")
print(f"Using device: {device}")

# Check HF token
hf_token = os.getenv('HF_TOKEN')
print(f"HF_TOKEN set: {hf_token is not None}")

---

## Part 1: Exploring the Model Hub

The [Hugging Face Model Hub](https://huggingface.co/models) hosts thousands of pre-trained models.

### Key Genomic Models

| Model | Organization | Parameters | Tokenization | Best For |
|-------|-------------|------------|--------------|----------|
| DNABERT-2 | Zhihan1996 | 117M | BPE | General DNA |
| Nucleotide Transformer | InstaDeep | 50M-2.5B | K-mer | Reference genomes |
| HyenaDNA | LongSafari | 1.4M-14M | Character | Long sequences |

### Exercise: Browse the Hub

1. Visit https://huggingface.co/models
2. Search for "DNABERT", "nucleotide transformer", "genomic"
3. Click on a model to view its **Model Card**

### What to Look For in a Model Card

| Section | What It Tells You |
|---------|-------------------|
| Model Description | Architecture, training objective |
| Intended Uses | What tasks it's designed for |
| Training Data | What sequences it saw during training |
| Limitations | Known weaknesses or biases |
| How to Use | Code examples |

In [ ]:
# Let's programmatically get model info
from huggingface_hub import HfApi

api = HfApi()

# Search for genomic models
print("Searching for genomic models on Hugging Face Hub...\n")

models_to_check = [
    "zhihan1996/DNABERT-2-117M",
    "InstaDeepAI/nucleotide-transformer-v2-50m-multi-species",
    "LongSafari/hyenadna-medium-160k-seqlen-hf"
]

for model_id in models_to_check:
    try:
        info = api.model_info(model_id)
        print(f"Model: {model_id}")
        print(f"  Downloads: {info.downloads:,}")
        print(f"  Likes: {info.likes}")
        print(f"  Tags: {info.tags[:5] if info.tags else 'N/A'}")
        print()
    except Exception as e:
        print(f"Could not fetch {model_id}: {e}\n")

---

## Part 2: Hugging Face Inference API

The **Inference API** lets you run models in the cloud without local setup.

### How It Works

```
Your Code  →  HTTP Request  →  HF Servers  →  Model Inference  →  Response
```

### When to Use API vs Local

| Scenario | Recommendation |
|----------|----------------|
| Quick prototyping | API |
| No GPU available | API |
| Large batch processing | Local |
| Production deployment | Local |
| Privacy-sensitive data | Local |

In [ ]:
def query_hf_api(sequence, model_id, token):
    """
    Query the Hugging Face Inference API.
    
    Args:
        sequence: Input DNA sequence (may contain <mask> for MLM)
        model_id: Hugging Face model identifier
        token: HF API token
    
    Returns:
        JSON response with predictions, or None if error
    """
    url = f"https://router.huggingface.co/hf-inference/models/{model_id}"
    headers = {"Authorization": f"Bearer {token}"}
    
    response = requests.post(url, headers=headers, json={"inputs": sequence})
    
    if response.status_code != 200:
        print(f"API Error {response.status_code}: {response.text[:200]}")
        return None
    
    try:
        return response.json()
    except Exception as e:
        print(f"JSON parse error: {e}")
        return None

### 2.1 Masked Language Modeling (Fill-in-the-Blank)

MLM asks the model to predict what nucleotides should fill a `<mask>` token. This demonstrates how the model has learned DNA "grammar".

In [ ]:
# Example: Ask the model to fill in the <mask> token
masked_seq = "ATCGATCG<mask>ATCG"
model_id = "InstaDeepAI/nucleotide-transformer-500m-human-ref"

print(f"Input sequence: {masked_seq}")
print(f"Model: {model_id}")
print("\nQuerying API...")

result = query_hf_api(masked_seq, model_id, hf_token)

if result:
    print("\nTop 5 predictions for <mask>:")
    for i, pred in enumerate(result[:5]):
        print(f"  {i+1}. '{pred['token_str']}' ({pred['score']*100:.1f}%)")
else:
    print("API call failed - check your HF_TOKEN")

### 2.2 Local vs API Inference Comparison

Let's compare the speed and results of local GPU inference vs the cloud API.

In [ ]:
import logging
logging.getLogger("transformers.modeling_utils").setLevel(logging.ERROR)

model_id = "InstaDeepAI/nucleotide-transformer-500m-human-ref"
print(f"Model: {model_id}")
print(f"Device: {device}\n")

# Load model locally
print("Loading model locally...")
tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
mlm_model = AutoModelForMaskedLM.from_pretrained(model_id, trust_remote_code=True).to(device).eval()

# Masked sequence
masked_seq = "ATCGATCG<mask>ATCG"
tokens = tokenizer([masked_seq], return_tensors="pt", padding=True, truncation=True).to(device)

# --- Local Inference ---
print("\n" + "="*50)
print("LOCAL INFERENCE")
print("="*50)

t0 = time.time()
with torch.no_grad():
    outputs = mlm_model(**tokens)
    logits = outputs.logits
local_time = time.time() - t0

# Get predictions
mask_idx = (tokens["input_ids"] == tokenizer.mask_token_id).nonzero(as_tuple=True)
mask_logits = logits[mask_idx]
probs = F.softmax(mask_logits, dim=-1)
top_k = torch.topk(probs, 5, dim=-1)

print(f"Time: {local_time:.3f}s")
print("\nTop 5 predictions:")
for i, (tok_id, score) in enumerate(zip(top_k.indices[0].tolist(), top_k.values[0].tolist())):
    token = tokenizer.decode([tok_id]).strip()
    print(f"  {i+1}. '{token}' ({score*100:.1f}%)")

# --- API Inference ---
print("\n" + "="*50)
print("API INFERENCE")
print("="*50)

t0 = time.time()
api_result = query_hf_api(masked_seq, model_id, hf_token)
api_time = time.time() - t0

if api_result:
    print(f"Time: {api_time:.3f}s")
    print("\nTop 5 predictions:")
    for i, r in enumerate(api_result[:5]):
        print(f"  {i+1}. '{r['token_str']}' ({r['score']*100:.1f}%)")

# Summary
print("\n" + "="*50)
print("SUMMARY")
print("="*50)
print(f"Local:  {local_time:.3f}s")
print(f"API:    {api_time:.3f}s")
print(f"Speedup: {api_time/local_time:.1f}x faster locally")

---

## Part 3: Datasets Hub

Hugging Face also hosts **datasets** for training and evaluation.

### Genomic Datasets Available

| Dataset | Tasks | Sequences |
|---------|-------|----------|
| InstaDeepAI/nucleotide_transformer_downstream_tasks | 18 classification tasks | 500K+ |
| Genomic Benchmarks | Promoters, enhancers, etc. | Varies |

In [ ]:
from datasets import load_dataset

# Load InstaDeep's benchmark dataset
print("Loading genomic benchmark dataset...")
ds = load_dataset("InstaDeepAI/nucleotide_transformer_downstream_tasks")

print(f"\nDataset structure:")
print(ds)

# Available tasks
tasks = np.unique(ds['train']['task'])
print(f"\nAvailable tasks ({len(tasks)}):")
for task in tasks:
    print(f"  - {task}")

In [ ]:
# Explore a specific task
from collections import Counter

TASK = "promoter_all"
ds_task = ds.filter(lambda x: x['task'] == TASK)

print(f"Task: {TASK}")
print(f"Train samples: {len(ds_task['train']):,}")
print(f"Test samples:  {len(ds_task['test']):,}")

# Class distribution
labels = [ex['label'] for ex in ds_task['train']]
print(f"\nClass distribution: {Counter(labels)}")

# Sample sequence
sample = ds_task['train'][0]
print(f"\nSample sequence (first 100bp):")
print(f"  {sample['sequence'][:100]}...")
print(f"  Label: {sample['label']}")

---

## Summary

In this notebook, you learned:

| Topic | Key Points |
|-------|------------|
| **Model Hub** | Browse models, read Model Cards, compare architectures |
| **Inference API** | Quick prototyping without local GPU |
| **Local vs API** | Local is faster; API is more convenient |
| **Datasets Hub** | Access benchmark datasets for training/evaluation |

---

## What's Next?

Tomorrow (Day 2), we'll put this all together:

1. **Fine-tune a classifier** on the promoter detection task
2. **Use PEFT/LoRA** for parameter-efficient fine-tuning

You now have all the building blocks:
- Encoding/tokenization (Day 1 AM)
- Model architecture understanding (Day 1 AM)
- Embedding extraction (Day 1 PM)
- HF ecosystem tools (Day 1 PM) ← You are here